# 🚀 Fine-Tuning Qwen2.5-Coder-0.5B-Instruct on NL2Bash (Unsloth + Colab T4)

Fine-tuning **Qwen2.5-Coder-0.5B-Instruct** using Unsloth on Google Colab.
- **Model**: `unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit`
- **VRAM Usage**: < 2 GB (Free T4 compatible)
- **Training Time**: ~10–15 minutes
- **Monitoring**: Live TensorBoard, Weights & Biases (WandB), & Real-time GPU stats
- **Export**: Merged 16-bit HuggingFace & Quantized GGUF (`q4_k_m`) for Ollama

### 0. Optional: Start SSH Server (For direct terminal control from your PC)

In [ ]:
# Run this cell if you want to SSH directly into this Colab from your local AGY terminal
!pip install -q colab_ssh
from colab_ssh import launch_ssh_cloudflared

# Set your password here
launch_ssh_cloudflared(password="colab12345")

### Step 1: Install Dependencies

In [ ]:
%%capture
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install unsloth
!pip install tensorboard wandb

### Step 2: Load Qwen2.5-Coder with 4-bit Quantization

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # Auto-detects float16 for T4 or bfloat16 for Ampere+
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Check initial GPU memory
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name} | Total VRAM: {gpu_stats.total_memory / 1024**3:.2f} GB")
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB | Reserved: {torch.cuda.memory_reserved() / 1024**3:.2f} GB")

### Step 3: Attach LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Optimized to 0 in Unsloth
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

### Step 4: Prepare the NL2Bash Dataset (ChatML format)

In [ ]:
from datasets import load_dataset

# Load clean natural language to Bash dataset
dataset = load_dataset("emirkaanozdemr/bash_command_data_6K", split="train")

def format_prompts(batch):
    prompts = batch.get("prompt") or batch.get("instruction")
    completions = batch.get("completion") or batch.get("command")
    formatted_texts = []
    for instruction, command in zip(prompts, completions):
        messages = [
            {
                "role": "system",
                "content": "You are a Linux and Ubuntu terminal assistant. Return only the exact, executable Bash command matching the user request with no markdown formatting or conversational filler."
            },
            {"role": "user", "content": str(instruction).strip()},
            {"role": "assistant", "content": str(command).strip()}
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_texts.append(text)
    return {"text": formatted_texts}

dataset = dataset.map(format_prompts, batched=True)

### Step 5: Launch Live TensorBoard Monitoring & Train

In [ ]:
# Load TensorBoard extension inside notebook for live loss & learning rate tracking
%load_ext tensorboard
%tensorboard --logdir outputs/logs

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Keeps short commands isolated
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 300, # or num_train_epochs = 1
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,  # Frequent logging for live monitoring
        logging_first_step = True,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        logging_dir = "outputs/logs",
        report_to = "tensorboard", # or "wandb"
    ),
)

trainer_stats = trainer.train()

### Step 6: Test Inference Inside Colab

In [ ]:
FastLanguageModel.for_inference(model)

test_prompt = [
    {
        "role": "system",
        "content": "You are a Linux and Ubuntu terminal assistant. Return only the exact, executable Bash command matching the user request with no markdown formatting or conversational filler."
    },
    {"role": "user", "content": "Find all processes running on port 8080 and terminate them immediately."}
]

inputs = tokenizer.apply_chat_template(
    test_prompt,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True)
print("Result:", tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))

### Step 7: Export to Merged 16-bit & GGUF (for Ollama)

In [ ]:
# Save 16-bit LoRA adapter (Hugging Face format)
model.save_pretrained_merged("models/qwen2.5-coder-0.5b-bash-merged", tokenizer, save_method="merged_16bit")

# Export directly to GGUF (q4_k_m is recommended for 0.5B)
model.save_pretrained_gguf("models/qwen2.5-coder-0.5b-bash-gguf", tokenizer, quantization_method="q4_k_m")

### Optional: Download or Save Directly to Google Drive

In [ ]:
# Save directly to Google Drive so it syncs to your PC automatically:
from google.colab import drive
drive.mount('/content/drive')
!cp -r models/qwen2.5-coder-0.5b-bash-gguf /content/drive/MyDrive/